# Example use of FlashGen with OpenAI module and self-signed certificate

**NOTE**: the self-signed certiciate authority bundle is **NOT** installed on the client system


In [1]:
%load_ext autoreload
%autoreload 2

In [12]:
# Gather API key from the .env file, create a client using the CA bundle, then run a simple query.

# Note: this is intentionally a self-contained example with simple asserts and prints
# Note: credentials are obfuscated by default, use debug=True to unmask

import os
from dotenv import load_dotenv
import httpx
from openai import OpenAI
from pathlib import Path


def get_server_fqdn(debug: bool = False) -> tuple[str, str]:
    """Obtain server FQDN from .env"""
    envvar_name = f"SERVER_FQDN"
    print(f"Get server FQDN: {envvar_name=}")

    load_dotenv(override=True)
    server_fqdn = os.getenv(envvar_name)
    assert server_fqdn and isinstance(server_fqdn, str) and (len(server_fqdn) > 0)
    assert "local" in server_fqdn
    server_hostname, local = server_fqdn.rsplit(".", 1)

    if debug:
        print(f"Get server FQDN: {server_fqdn=}")
    else:
        # obfuscate
        print(f"Get server FQDN={'*' * len(server_hostname)}.{local}")
    return server_fqdn, server_hostname


def get_api_key(service: str | None = None, debug: bool = False):
    """Obtain API key from .env"""
    service = service or "OLLAMA"
    envvar_name = f"{service}_API_KEY"
    print(f"Get API key: {envvar_name=}")

    load_dotenv(override=True)
    api_key = os.getenv(envvar_name)
    assert api_key and isinstance(api_key, str) and (len(api_key) > 0)
    if debug:
        print(f"Get API key: {api_key=}")
    else:
        print(f"Get API key: api_key={'*' * len(api_key)}")
    return api_key


def get_self_signed_client(
    ollama_api_key: str, flashgen_server_fqdn: str, ollama_port: int = 4443, debug=False
) -> OpenAI:
    """Connect to ollama via traefik with self-signed TLS certificate."""
    ca_bundle = f"{flashgen_server_fqdn}.pem"
    assert Path(ca_bundle).exists()

    base_url = f"https://{flashgen_server_fqdn}:{ollama_port}/ollama/v1/"

    if debug:
        print(f"Get client: {flashgen_server_fqdn=}")
        print(f"Get client: {base_url=}")
        print(f"Get client: {ca_bundle=}")
    else:
        server_hostname, local = flashgen_server_fqdn.rsplit(".", 1)
        server_hostname_obfuscated = "*" * len(server_hostname)
        replace = (server_hostname, server_hostname_obfuscated)
        print(f"Get client: flashgen_server_fqdn={flashgen_server_fqdn.replace(*replace)}")
        print(f"Get client: base_url={base_url.replace(*replace)}")
        print(f"Get client: ca_bundle={ca_bundle.replace(*replace)}")

    http_client = httpx.Client(verify=ca_bundle)
    ollama_client = OpenAI(api_key=ollama_api_key, base_url=base_url, http_client=http_client)

    # simple check for validity
    # will raise various exceptions (e.g., AuthenticationError) if not configured correctly
    _ = ollama_client.models.list()

    return ollama_client


def run_simple_query(ollama_client: OpenAI, query: str | None = None, model: str | None = None) -> str:
    """Run a simple query (default is 'Say Hello')."""
    model = model or "qwen2.5:1.5b-instruct"
    query = query or "Say Hello from Ollama"

    # Note: this must be a list of dictionaries
    messages = [{"role": "user", "content": "Say Hello from Ollama"}]
    model = "qwen2.5:1.5b-instruct"
    print(f"Run query: {model=}")
    print(f"Run query: {query=}")

    assert isinstance(messages, list)
    assert isinstance(messages[0], dict)
    response = ollama_client.chat.completions.create(model=model, messages=messages)
    answer = response.choices[0].message.content
    print(f"Run query: {answer=}")
    return answer


debug = False
flashgen_server_fqdn, flashgen_hostname = get_server_fqdn(debug=debug)
ollama_api_key = get_api_key(debug=debug)
ollama_client = get_self_signed_client(
    ollama_api_key=ollama_api_key, flashgen_server_fqdn=flashgen_server_fqdn, debug=debug
)
response = run_simple_query(ollama_client=ollama_client)

print("")
print(response)

Get server FQDN: envvar_name='SERVER_FQDN'
Get server FQDN=*******.local
Get API key: envvar_name='OLLAMA_API_KEY'
Get API key: api_key=********************************************
Get client: flashgen_server_fqdn=*******.local
Get client: base_url=https://*******.local:4443/ollama/v1/
Get client: ca_bundle=*******.local.pem
Run query: model='qwen2.5:1.5b-instruct'
Run query: query='Say Hello from Ollama'
Run query: answer="Hello there! It seems like something is yelling in the chatbox. This might have been an error caused by too much excitement over certain words or the system's inability to interpret text correctly. Normally, I'm a very gentle and helpful assistant here, but sometimes it gets tricky! Is there anything more positive you'd like me to say?"

Hello there! It seems like something is yelling in the chatbox. This might have been an error caused by too much excitement over certain words or the system's inability to interpret text correctly. Normally, I'm a very gentle and he